# K-best and RFE: which inputs should logistic regression keep?

**Features** are inputs such as HP or move frequency; **hyperparameters** are choices such as C or k. This experiment selects features while holding LR C=.01 and other model settings fixed.

K-best scores each feature individually with a classification F-test. RFE fits LR repeatedly and removes small-coefficient features. Neither method guarantees higher accuracy. [Official scikit-learn guide](https://scikit-learn.org/stable/modules/feature_selection.html).

Same 7,996 development rows and five grouped folds as notebooks 06–08; earlier holdout and duplicates excluded. Vocabulary, scaling, constant removal and selection fit on each training fold only. Default: review saved evidence. Enable the flag below to repeat all six candidates.


In [ ]:
from pathlib import Path
import sys, csv, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pokemon_challenge").is_dir())
sys.path.insert(0, str(ROOT / "notebooks/leonard"))
RUN_NEW_EXPERIMENT = False
RUN_FOLDER = ROOT / "outputs/leonard/20261102_144359_081808_feature_selection"
if RUN_NEW_EXPERIMENT:
    from run_feature_selection_experiment import run_selection
    RUN_FOLDER = run_selection()
print("Artifacts:", RUN_FOLDER)


## How to set up selection

The runner builds the following pipeline: `identity encoding → numeric scaling → constant removal → selector → logistic regression`. The selector sees training labels only. It must not be fitted once on the whole dataset before cross-validation.

```python
from run_feature_selection_experiment import make_selection_model

kbest_model = make_selection_model(columns, method="kbest", k=300)
rfe_model = make_selection_model(columns, method="rfe", k=300)

# Inside each fold:
kbest_model.fit(X_train, y_train)
predictions = kbest_model.predict(X_validation)
```

`columns` is the recorded baseline numeric column list; the raw input frame also contains identity dictionaries and categorical strings, as prepared by the runner. RFE removes 25% of the initial feature count per iteration for reasonable runtime.


In [ ]:
with (RUN_FOLDER / "cv_results.csv").open() as stream:
    rows = list(csv.DictReader(stream))
for row in rows:
    print(f"{row['candidate']:16s} {float(row['mean_selected_features']):7.0f} features  {100*float(row['cv_accuracy']):.3f}%")


## Decision

Keep the previous full LR model: **67.259%**. Best K-best score: **66.921%** (300). RFE300/1000: **67.159%**, only 0.100 pp lower. RFE300 retains about 7% of the inputs and is worth noting as a compact alternative, but it did not improve accuracy. Its OOF predictions include eight fewer correct answers out of 7,996. This small difference is not an independent significance result.

The recommended CSV is unchanged, byte-for-byte. No holdout re-evaluation or Kaggle upload. Selection lists vary between folds; inspect the actual feature names below. Stability frequencies also reflect features absent from some training vocabularies. RFE coefficients depend on scaling, C and correlated features.


In [ ]:
with (RUN_FOLDER / "selection_stability.csv").open() as stream:
    stable = [row for row in csv.DictReader(stream) if row["candidate"] == "rfe_300" and int(row["selected_folds"]) == 5]
print(f"RFE300 features selected in all five folds: {len(stable)}")
for row in stable[:20]:
    print(row["feature"])
print("Full lists:", RUN_FOLDER / "selected_features_by_fold.csv")
print(json.dumps(json.loads((RUN_FOLDER / "validation_report.json").read_text()), indent=2))
